## Experimenting with Data processing -> is formally in `data-frame-builder.py` 
### this is just for visualistaiont & intution, will become a data loader class

In [37]:
# create dataset which includes all the macro data processed
import pandas as pd
import os

current_dir = os.getcwd()
nb_path = os.path.dirname(current_dir)
repo_path = os.path.dirname(nb_path)
print(repo_path)
type(repo_path)

/Users/minna/Code/FS26/AML/aml2026-group-3


str

In [2]:
# ------ get daily data ---------
daily_path = repo_path + '/data/macro-vars-daily.csv'
# print(daily_path)

df_daily = pd.read_csv(daily_path)
df_daily = df_daily.rename(columns = {'Unnamed: 0':'date'})
df_daily['date'] = pd.to_datetime(df_daily['date'])
df_daily = df_daily.drop(columns=["SOFR", "T10Y2Y", "EUR"]) # remove shorter vars: SOFR, T10Y2Y, EUR
# print(df_daily.tail(20))

# ------- get quarterly data --------

qrtly_path = repo_path + '/data/macro-vars-quarterly.csv'
df_quartly = pd.read_csv(qrtly_path)
df_quartly = df_quartly.rename(columns = {'Unnamed: 0':'date'})
df_quartly['date'] = pd.to_datetime(df_quartly['date'])
# print(df_quartly.tail(10))

# --------- get monthly data ---------
monthly_path = repo_path + '/data/macro-vars-monthly.csv'
df_monthly = pd.read_csv(monthly_path)
df_monthly = df_monthly.rename(columns = {'Unnamed: 0':'date'})
# remove columns which are too short 
df_monthly = df_monthly.drop(columns=["PCEPI", "JTSJOL", "UMCSENT"])

df_monthly['date'] = pd.to_datetime(df_monthly['date'])
# print(df_monthly.head(10))

In [3]:
# merge 
df = pd.merge(df_monthly, df_quartly, on='date', how='left')
df = pd.merge(df_daily, df, on='date', how='left') 

# forward-fill monthly/quarterly vars: repeat last published value until updated
monthly_quarterly_cols = ["CPI", "PAYEMS", "INDPRO", "UNRATE", "GDP"]
df[monthly_quarterly_cols] = df[monthly_quarterly_cols].ffill()

# trim leading rows before daily vars (GBP, YEN) begin
start_idx = df['GBP'].first_valid_index()
df = df.iloc[start_idx:].reset_index(drop=True)

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}, {len(df)} rows")
print(df.head(10))


Date range: 1971-01-04 00:00:00 to 2026-04-10 00:00:00, 20186 rows
        date     GBP     YEN   FFR   CPI   PAYEMS   INDPRO  UNRATE       GDP
0 1971-01-04  2.3938  357.73  5.50  39.9  70865.0  37.5753     5.9  1135.156
1 1971-01-05  2.3949  357.81  5.50  39.9  70865.0  37.5753     5.9  1135.156
2 1971-01-06  2.3967  357.86  3.75  39.9  70865.0  37.5753     5.9  1135.156
3 1971-01-07  2.3963  357.87  4.50  39.9  70865.0  37.5753     5.9  1135.156
4 1971-01-08  2.3972  357.82  4.50  39.9  70865.0  37.5753     5.9  1135.156
5 1971-01-09     NaN     NaN  4.50  39.9  70865.0  37.5753     5.9  1135.156
6 1971-01-10     NaN     NaN  4.50  39.9  70865.0  37.5753     5.9  1135.156
7 1971-01-11  2.3992  357.95  4.38  39.9  70865.0  37.5753     5.9  1135.156
8 1971-01-12  2.4001  358.06  4.25  39.9  70865.0  37.5753     5.9  1135.156
9 1971-01-13  2.4021  358.44  3.25  39.9  70865.0  37.5753     5.9  1135.156


In [4]:
print(df.tail(10))

            date     GBP     YEN   FFR      CPI    PAYEMS    INDPRO  UNRATE  \
20176 2026-04-01  1.3331  158.63  3.64  330.293  158637.0  101.7898     4.3   
20177 2026-04-02  1.3241  159.34  3.64  330.293  158637.0  101.7898     4.3   
20178 2026-04-03  1.3202  159.64  3.64  330.293  158637.0  101.7898     4.3   
20179 2026-04-04     NaN     NaN  3.64  330.293  158637.0  101.7898     4.3   
20180 2026-04-05     NaN     NaN  3.64  330.293  158637.0  101.7898     4.3   
20181 2026-04-06     NaN     NaN  3.64  330.293  158637.0  101.7898     4.3   
20182 2026-04-07     NaN     NaN  3.64  330.293  158637.0  101.7898     4.3   
20183 2026-04-08     NaN     NaN  3.64  330.293  158637.0  101.7898     4.3   
20184 2026-04-09     NaN     NaN  3.64  330.293  158637.0  101.7898     4.3   
20185 2026-04-10     NaN     NaN   NaN  330.293  158637.0  101.7898     4.3   

             GDP  
20176  31422.526  
20177  31422.526  
20178  31422.526  
20179  31422.526  
20180  31422.526  
20181  31422.526

### create CV code, which splits it into different folds

In [5]:
N_FOLDS = 1 # holdout cv
FINAL_HOLDOUT = 120 # 4 months
# data without holdout
DATA_CV_LEN = len(df) - FINAL_HOLDOUT
# for one fold
FOLD_TRAIN = int((0.8*DATA_CV_LEN)/N_FOLDS)
FOLD_TEST = int((0.2*DATA_CV_LEN)/N_FOLDS)

In [6]:
print(DATA_CV_LEN)
print(FOLD_TRAIN)
print(FOLD_TEST)

20066
16052
4013


In [7]:
def generate_splits(df, n_folds=N_FOLDS, fold_train=FOLD_TRAIN, fold_test=FOLD_TEST, final_holdout=FINAL_HOLDOUT):
    """Expanding-window CV splits. Returns list of fold dicts + final holdout df."""
    df_cv = df.iloc[:-final_holdout].reset_index(drop=True)
    # first remove final holdout, not part of cv!
    df_holdout = df.iloc[-final_holdout:].reset_index(drop=True)

    splits = []
    # rolling window cv
    for i in range(n_folds):
        train_end = (i + 1) * fold_train
        test_end = train_end + fold_test
        splits.append({
            "fold": i + 1,
            "train": df_cv.iloc[:train_end],
            "test": df_cv.iloc[train_end:test_end],
        })

    return splits, df_holdout


### based on these splits, feed to new models

In [8]:
splits, holdout = generate_splits(df)
# print(splits[0]) # fold 0 out of 4

fold_0 = splits[0]
print(fold_0['train']) # from 1971-01-04
# print(fold_0['test'])

            date     GBP     YEN   FFR      CPI    PAYEMS    INDPRO  UNRATE  \
0     1971-01-04  2.3938  357.73  5.50   39.900   70865.0   37.5753     5.9   
1     1971-01-05  2.3949  357.81  5.50   39.900   70865.0   37.5753     5.9   
2     1971-01-06  2.3967  357.86  3.75   39.900   70865.0   37.5753     5.9   
3     1971-01-07  2.3963  357.87  4.50   39.900   70865.0   37.5753     5.9   
4     1971-01-08  2.3972  357.82  4.50   39.900   70865.0   37.5753     5.9   
...          ...     ...     ...   ...      ...       ...       ...     ...   
16047 2014-12-11  1.5712  119.40  0.12  236.252  140364.0  103.7044     5.6   
16048 2014-12-12  1.5728  118.22  0.12  236.252  140364.0  103.7044     5.6   
16049 2014-12-13     NaN     NaN  0.12  236.252  140364.0  103.7044     5.6   
16050 2014-12-14     NaN     NaN  0.12  236.252  140364.0  103.7044     5.6   
16051 2014-12-15  1.5663  117.75  0.11  236.252  140364.0  103.7044     5.6   

             GDP  
0       1135.156  
1       1135.

In [9]:
splits, holdout = generate_splits(df)

for s in splits:
    tr, te = s["train"], s["test"]
    print(f"Fold {s['fold']}: train [{tr['date'].min().date()} – {tr['date'].max().date()}] ({len(tr)} rows) | "
          f"test [{te['date'].min().date()} – {te['date'].max().date()}] ({len(te)} rows)")

print(f"Holdout: [{holdout['date'].min().date()} – {holdout['date'].max().date()}] ({len(holdout)} rows)")

Fold 1: train [1971-01-04 – 2014-12-15] (16052 rows) | test [2014-12-16 – 2025-12-10] (4013 rows)
Holdout: [2025-12-12 – 2026-04-10] (120 rows)


TODO: this is a bit shitty, because the test period includes covid, so all models will perform worse because of it... in future, maybe multiple folds

In [ ]:
TARGET_COLS = ["CPI", "PAYEMS", "INDPRO", "UNRATE", "GDP"]
QUARTERLY_TARGETS = {"GDP"}          # native quarterly frequency
MONTHLY_TARGETS   = {"CPI", "PAYEMS", "INDPRO", "UNRATE"}

def get_fold_data(splits, fold: int, train: bool = True, model: str = "TFT", target: str = "CPI"):
    """
    fold   : 0-indexed (0–4)
    train  : True → training split, False → test split
    model  : "AR" or "ARIMA" → univariate series resampled to native frequency
             "TFT"           → full daily dataframe
    target : which macro variable to forecast (only used for AR / ARIMA)
    """
    assert target in TARGET_COLS, f"target must be one of {TARGET_COLS}"
    df_split = splits[fold]["train" if train else "test"]

    if model in ("AR", "ARIMA"):
        freq = "QS" if target in QUARTERLY_TARGETS else "MS"
        return (df_split.set_index("date")[[target]]
                        .resample(freq).first()
                        .dropna()
                        .reset_index())

    return df_split.copy()  # TFT: full daily df

In [32]:
# quick sanity check
tft_train = get_fold_data(splits, fold=0, train=True,  model="TFT")
tft_test   = get_fold_data(splits, fold=0, train=False, model="TFT", target= "CPI")
ar_train  = get_fold_data(splits, fold=0, train=True,  model="AR",  target="CPI")
ar_test   = get_fold_data(splits, fold=0, train=False, model="ARIMA",  target="GDP")

print(f"TFT train : {len(tft_train)} daily rows   | cols: {list(tft_train.columns)}")
print(f"TFT test : {len(tft_test)} daily rows   | cols: {list(tft_test.columns)}")
print(f"AR  train : {len(ar_train)} monthly rows | cols: {list(ar_train.columns)}")
print(f"ARIMA  test  : {len(ar_test)} quarterly rows  | cols: {list(ar_test.columns)}")

TFT train : 16052 daily rows   | cols: ['date', 'GBP', 'YEN', 'FFR', 'CPI', 'PAYEMS', 'INDPRO', 'UNRATE', 'GDP']
TFT test : 4013 daily rows   | cols: ['date', 'GBP', 'YEN', 'FFR', 'CPI', 'PAYEMS', 'INDPRO', 'UNRATE', 'GDP']
AR  train : 528 monthly rows | cols: ['date', 'CPI']
ARIMA  test  : 45 quarterly rows  | cols: ['date', 'GDP']


In [ ]:
print(ar_test) # ARIMA test, gdp var

         date        GDP
0  2014-10-01  17912.079
1  2015-01-01  18063.529
2  2015-04-01  18279.784
3  2015-07-01  18401.626
4  2015-10-01  18435.137
5  2016-01-01  18525.933
6  2016-04-01  18711.702
7  2016-07-01  18892.639
8  2016-10-01  19089.379
9  2017-01-01  19280.084
10 2017-04-01  19438.643
11 2017-07-01  19692.595
12 2017-10-01  20037.088
13 2018-01-01  20328.553
14 2018-04-01  20580.912
15 2018-07-01  20798.730
16 2018-10-01  20917.867
17 2019-01-01  21111.600
18 2019-04-01  21397.938
19 2019-07-01  21717.171
20 2019-10-01  21933.217
21 2020-01-01  21751.238
22 2020-04-01  19958.291
23 2020-07-01  21704.437
24 2020-10-01  22087.160
25 2021-01-01  22680.693
26 2021-04-01  23425.910
27 2021-07-01  23982.379
28 2021-10-01  24813.600
29 2022-01-01  25250.347
30 2022-04-01  25861.292
31 2022-07-01  26336.304
32 2022-10-01  26770.514
33 2023-01-01  27216.445
34 2023-04-01  27530.055
35 2023-07-01  28074.846
36 2023-10-01  28424.722
37 2024-01-01  28708.161
38 2024-04-01  29147.044


In [36]:
type(ar_test)

pandas.DataFrame

### next steps: create a .py file with the data loader class! 
